<a href="https://colab.research.google.com/github/anamacao/FAPESP-PIBIC-scrapping/blob/main/UE/uniaoEuropeia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importações e configuração

In [ ]:
# ==============================================================
# UE Scraper – Versão Notebook (SEM BD)
# ==============================================================

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from datetime import datetime
import time
import plotly.express as px
import plotly.io as pio

# Gráficos interativos na célula
pio.renderers.default = "notebook_connected"

HEADERS = {
    "User-Agent":
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}

print("Pronto para scraping da UE!")

## Funções auxiliares

In [ ]:
# ==============================
# FUNÇÕES AUXILIARES
# ==============================

def carregar_pagina_selenium(url):
    """Carrega a página HTML sem depender de Chrome/ChromeDriver."""
    response = requests.get(url, headers=HEADERS, timeout=20)
    response.raise_for_status()
    return BeautifulSoup(response.content, "html.parser")


def extrair_paragrafos(soup):
    """Extrai <p> com pelo menos 12 palavras ou que terminam com ponto."""
    paragrafos = []
    for p in soup.find_all("p"):
        txt = p.get_text(strip=True)
        if txt and (len(txt.split()) >= 12 or txt.endswith(".")):
            paragrafos.append(txt)
    return paragrafos or ["NA"]

## Scraper principal (todas as páginas)

In [ ]:
# ==============================
# SCRAPER UE – COM URL CORRIGIDA
# ==============================

def scrape_ue(max_page=5):
    print(f"Coletando as {max_page} páginas mais recentes …")
    start = time.time()
    articles = []
    base = "https://european-union.europa.eu/news-and-events/featured-news_en?page="

    for pg in range(max_page):
        url = base + str(pg)
        print(f"\nPágina {pg}: {url}")

        try:
            r = requests.get(url, headers=HEADERS, timeout=20)
            r.raise_for_status()
            soup = BeautifulSoup(r.content, "html.parser")

            itens = soup.select("article.ecl-content-item")
            if not itens:
                raise RuntimeError("Página sem artigos: o seletor da fonte mudou")
            print(f"   {len(itens)} notícias.")

            for item in itens:
                titulo_tag = item.find("div", class_="ecl-content-block__title")
                titulo = titulo_tag.a.get_text(strip=True) if titulo_tag else "NA"

                # === CORREÇÃO DA URL ===
                href = titulo_tag.a["href"] if titulo_tag and titulo_tag.a else "NA"
                if href.startswith("/"):
                    link = "https://european-union.europa.eu" + href
                elif href.startswith("http"):
                    link = href
                else:
                    link = "https://european-union.europa.eu/" + href
                # === FIM DA CORREÇÃO ===

                data_iso = item.find("time")["datetime"] if item.find("time") else None
                data = "NA"
                if data_iso:
                    try:
                        data = datetime.fromisoformat(data_iso.split("T")[0]).strftime("%d/%m/%Y")
                    except:
                        data = data_iso.split("T")[0]

                horario = data_iso.split("T")[1].split("Z")[0] if data_iso and "T" in data_iso else "NA"

                cat_tag = item.find("li", class_="ecl-content-block__primary-meta-item")
                tags = [cat_tag.get_text(strip=True)] if cat_tag else []

                img_tag = item.find("img")
                imagem = img_tag["src"] if img_tag and img_tag.get("src") else "NA"
                imagens = [{"Imagem": imagem}] if imagem != "NA" else []

                descricao = item.select_one(".ecl-content-block__description")
                parags = extrair_paragrafos(descricao) if descricao else ["NA"]

                articles.append({
                    "titulo": titulo,
                    "data": data,
                    "horario": horario,
                    "link": link,
                    "tags": " | ".join(tags),
                    "imagem": imagem,
                    "paragrafos": " || ".join(parags[:5]) + ("..." if len(parags) > 5 else ""),
                    "source": "UE News"
                })

            time.sleep(1.0)

        except Exception as e:
            print(f"   Erro na página {pg}: {e}")
            break

    df = pd.DataFrame(articles, columns=["titulo", "data", "horario", "link", "tags", "imagem", "paragrafos", "source"])
    print(f"\n{len(df)} notícias coletadas em {time.time()-start:.1f}s")
    display(df.head(20))
    return df

# trocar para numero necessario
df_ue = scrape_ue(max_page=2)

## Filtro + Gráficos interativos

In [ ]:
# ==============================
# FILTRO & GRÁFICOS (100% FUNCIONAL)
# ==============================

keywords = ['digital', 'governança', 'internet', 'IA', 'privacidade', 'regulação']
pattern = r'|'.join(keywords)

df_filt = df_ue[
    df_ue['titulo'].str.contains(pattern, case=False, na=False, regex=True) |
    df_ue['paragrafos'].str.contains(pattern, case=False, na=False, regex=True)
].copy()

print(f"\n{len(df_filt)} notícias filtradas (de {len(df_ue)})")

def plot_charts(df):
    if df.empty:
        print("Sem dados para gráficos.")
        return

    # 1. Top 15
    top15 = df.head(15).copy()
    top15['rank'] = range(1, len(top15)+1)
    fig1 = px.bar(top15, x='rank', y='titulo', orientation='h',
                  title='Top 15 Notícias')
    fig1.update_layout(height=600,
                       yaxis={'categoryorder':'array',
                              'categoryarray':top15['titulo'][::-1]})
    fig1.show()

    # 2. Pizza por tag — CORRIGIDO
    tag_series = df['tags'].str.split(r' \| ').explode()
    if not tag_series.empty:
        tag_counts = tag_series.value_counts().head(10).reset_index()
        tag_counts.columns = ['tag', 'count']  # ← nomes corretos
        fig2 = px.pie(tag_counts, names='tag', values='count', title='Distribuição de Tags')
        fig2.show()
    else:
        print("Nenhuma tag para exibir no gráfico de pizza.")

    # 3. Nuvem de palavras
    text = ' '.join(df['titulo'] + ' ' + df['paragrafos']).lower()
    words = re.findall(r'\b\w{4,}\b', text)
    if words:
        wc = pd.Series(words).value_counts().head(20).reset_index()
        wc.columns = ['palavra', 'freq']
        fig3 = px.treemap(wc, path=['palavra'], values='freq',
                          title='Nuvem de Palavras')
        fig3.show()
    else:
        print("Nenhuma palavra para nuvem.")

plot_charts(df_filt if not df_filt.empty else df_ue)

## Esqueleto do banco (só referência)

In [ ]:
# ==============================
# ESQUELETO DO BANCO (NÃO EXECUTADO)
# ==============================
"""
import sqlite3
DB = 'ue_news.db'
def init_db():
    conn = sqlite3.connect(DB)
    conn.execute('''
        CREATE TABLE IF NOT EXISTS news (
            id INTEGER PRIMARY KEY,
            titulo TEXT, data TEXT, horario TEXT, link TEXT UNIQUE,
            tags TEXT, imagem TEXT, paragrafos TEXT
        )
    ''')
    conn.close()
"""
